# Visual Analytics: NYPD Motor Vehicle Collisions Analysis
## Uncovering Patterns in NYC Traffic Safety

**Author:** Mayank Waghmare  
**Dataset:** NYPD Motor Vehicle Collisions Sample (3,000 records)  
**Objective:** Explore collision patterns, identify high-risk factors, and provide data-driven insights for traffic safety improvements in New York City.


## Introduction

Traffic collisions are a significant public safety concern in New York City. This analysis explores a sample of motor vehicle collision data to understand:

* **When and where** collisions occur most frequently
* **What factors** contribute to collisions
* **Who is affected** - pedestrians, cyclists, or motorists
* **How severe** the collisions are across different boroughs

Through visual analytics using ggplot2, we'll uncover patterns that can inform traffic safety interventions and policy decisions.


## Setup and Data Loading


In [ ]:
# Install packages with proper dependency resolution
install.packages("vctrs", repos = "https://cloud.r-project.org/", quiet = TRUE)
install.packages("rlang", repos = "https://cloud.r-project.org/", quiet = TRUE)
install.packages("dplyr", repos = "https://cloud.r-project.org/", quiet = TRUE)
install.packages("ggplot2", repos = "https://cloud.r-project.org/", quiet = TRUE)
install.packages("tidyr", repos = "https://cloud.r-project.org/", quiet = TRUE)
install.packages("lubridate", repos = "https://cloud.r-project.org/", quiet = TRUE)
install.packages("scales", repos = "https://cloud.r-project.org/", quiet = TRUE)

# Load libraries
library(ggplot2)
library(dplyr)
library(tidyr)
library(lubridate)
library(scales)

# Set theme
theme_set(theme_minimal())

print("All libraries loaded successfully!")

In [ ]:
# Load the collision data
collisions <- read.csv('NYPD_Motor_Vehicle_Collisions_Sample.csv', stringsAsFactors = FALSE)

# Display basic information
cat("Dataset dimensions:", nrow(collisions), "rows and", ncol(collisions), "columns\n")
cat("\nFirst few column names:\n")
print(head(names(collisions), 10))

# Display structure
str(collisions[,1:8])

## Data Preprocessing


In [ ]:
# Clean and preprocess the data
collisions_clean <- collisions %>%
  mutate(
    # Parse date - handle different formats
    crash_date = as.Date(CRASH.DATE, format = "%m/%d/%Y"),
    
    # Extract hour from time
    crash_hour = as.numeric(substr(CRASH.TIME, 1, 2)),
    
    # Extract year, month, weekday
    crash_year = as.numeric(format(crash_date, "%Y")),
    crash_month = factor(months(crash_date, abbreviate = TRUE), 
                         levels = c("Jan", "Feb", "Mar", "Apr", "May", "Jun",
                                    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec")),
    crash_wday = factor(weekdays(crash_date, abbreviate = TRUE),
                        levels = c("Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun")),
    
    # Create time of day categories
    time_of_day = case_when(
      crash_hour >= 6 & crash_hour < 12 ~ "Morning (6AM-12PM)",
      crash_hour >= 12 & crash_hour < 18 ~ "Afternoon (12PM-6PM)",
      crash_hour >= 18 & crash_hour <= 23 ~ "Evening (6PM-12AM)",
      TRUE ~ "Night (12AM-6AM)"
    ),
    
    # Total casualties
    total_injured = as.numeric(NUMBER.OF.PERSONS.INJURED),
    total_killed = as.numeric(NUMBER.OF.PERSONS.KILLED),
    total_casualties = total_injured + total_killed,
    
    # Borough
    borough = ifelse(is.na(BOROUGH) | BOROUGH == "", "Unknown", as.character(BOROUGH)),
    
    # Contributing factor
    contributing_factor = case_when(
      is.na(CONTRIBUTING.FACTOR.VEHICLE.1) ~ "Unspecified",
      CONTRIBUTING.FACTOR.VEHICLE.1 == "" ~ "Unspecified",
      CONTRIBUTING.FACTOR.VEHICLE.1 == "Unspecified" ~ "Unspecified",
      TRUE ~ as.character(CONTRIBUTING.FACTOR.VEHICLE.1)
    ),
    
    # Vehicle type
    vehicle_type = case_when(
      is.na(VEHICLE.TYPE.CODE.1) ~ "Unknown",
      VEHICLE.TYPE.CODE.1 == "" ~ "Unknown",
      TRUE ~ as.character(VEHICLE.TYPE.CODE.1)
    )
  )

# Remove rows with missing dates
collisions_clean <- collisions_clean %>% filter(!is.na(crash_date))

cat("Data preprocessing complete!\n")
cat("Clean dataset:", nrow(collisions_clean), "rows\n")
cat("Date range:", as.character(min(collisions_clean$crash_date, na.rm = TRUE)), 
    "to", as.character(max(collisions_clean$crash_date, na.rm = TRUE)), "\n")

---
# Part 1: Temporal Analysis
## When do collisions happen?


### Question 1: What is the hourly distribution of collisions throughout the day?


In [ ]:
# Hourly collision distribution
ggplot(collisions_clean, aes(x = crash_hour)) +
  geom_histogram(binwidth = 1, fill = "steelblue", color = "black", alpha = 0.7) +
  scale_x_continuous(breaks = seq(0, 23, 2)) +
  labs(
    title = "Collision Distribution Throughout the Day",
    subtitle = "Peak collision hours occur during evening rush hour (3-6 PM)",
    x = "Hour of Day (0-23)",
    y = "Number of Collisions"
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    plot.subtitle = element_text(color = "gray40", size = 10)
  )

**Key Insight:** The histogram reveals collision patterns align with traffic volume - lowest during early morning hours (2-5 AM) and peaking during afternoon rush hour.


### Question 2: How do collision patterns vary by time of day across different days of the week?


In [ ]:
# Collisions by time of day and day of week
time_day_summary <- collisions_clean %>%
  filter(!is.na(crash_wday)) %>%
  group_by(crash_wday, time_of_day) %>%
  summarise(count = n(), .groups = 'drop') %>%
  mutate(time_of_day = factor(time_of_day, 
                               levels = c("Morning (6AM-12PM)", "Afternoon (12PM-6PM)",
                                          "Evening (6PM-12AM)", "Night (12AM-6AM)")))

ggplot(time_day_summary, aes(x = crash_wday, y = count, fill = time_of_day)) +
  geom_bar(stat = "identity", position = "dodge") +
  scale_fill_manual(values = c("#66c2a5", "#fc8d62", "#8da0cb", "#e78ac3")) +
  labs(
    title = "Collision Patterns by Day of Week and Time of Day",
    subtitle = "Weekdays show consistent patterns; weekends differ",
    x = "Day of Week",
    y = "Number of Collisions",
    fill = "Time of Day"
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    legend.position = "bottom",
    axis.text.x = element_text(angle = 45, hjust = 1)
  )

**Key Insight:** Weekday collisions peak during commute times, while weekend patterns are more evenly distributed.


---
# Part 2: Geographic Analysis
## Where do collisions occur?


### Question 3: Which NYC boroughs have the highest collision rates?


In [ ]:
# Borough collision comparison
borough_summary <- collisions_clean %>%
  filter(borough != "Unknown") %>%
  group_by(borough) %>%
  summarise(total_collisions = n(), .groups = 'drop') %>%
  arrange(desc(total_collisions))

ggplot(borough_summary, aes(x = reorder(borough, total_collisions), y = total_collisions)) +
  geom_bar(stat = "identity", fill = "coral", alpha = 0.8) +
  geom_text(aes(label = total_collisions), hjust = -0.2, size = 4) +
  coord_flip() +
  labs(
    title = "Collision Frequency by NYC Borough",
    subtitle = "Brooklyn leads with the highest number of reported collisions",
    x = "Borough",
    y = "Number of Collisions"
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(face = "bold", size = 14)
  )

**Key Insight:** Brooklyn and Queens account for the majority of collisions, correlating with their population density.


### Question 4: How severe are collisions across different boroughs?


In [ ]:
# Borough severity analysis
collisions_with_casualties <- collisions_clean %>%
  filter(borough != "Unknown", total_casualties > 0)

ggplot(collisions_with_casualties, aes(x = reorder(borough, total_casualties, FUN = median), 
                                        y = total_casualties, fill = borough)) +
  geom_boxplot(alpha = 0.7, outlier.color = "red", outlier.size = 2) +
  scale_fill_manual(values = c("#fdb462", "#80b1d3", "#fb8072", "#bebada", "#b3de69")) +
  labs(
    title = "Collision Severity Distribution by Borough",
    subtitle = "Most collisions result in 1-2 casualties",
    x = "Borough",
    y = "Total Casualties (Injured + Killed)"
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    legend.position = "none"
  )

**Key Insight:** Collision severity is relatively consistent across boroughs, with most incidents resulting in 1-2 casualties.


---
# Part 3: Contributing Factors Analysis
## What causes collisions?


### Question 5: What are the most common contributing factors?


In [ ]:
# Top contributing factors
top_factors <- collisions_clean %>%
  group_by(contributing_factor) %>%
  summarise(count = n(), .groups = 'drop') %>%
  arrange(desc(count)) %>%
  head(10)

ggplot(top_factors, aes(x = reorder(contributing_factor, count), y = count)) +
  geom_bar(stat = "identity", fill = "darkgreen", alpha = 0.7) +
  geom_text(aes(label = count), hjust = -0.2, size = 3.5) +
  coord_flip() +
  labs(
    title = "Top 10 Contributing Factors to Collisions",
    subtitle = "Driver inattention/distraction is the leading cause",
    x = "Contributing Factor",
    y = "Number of Collisions"
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    axis.text.y = element_text(size = 9)
  )

**Key Insight:** Driver inattention/distraction is the most significant controllable factor.


### Question 6: How do factors relate to severity?


In [ ]:
# Factor severity analysis
factor_severity <- collisions_clean %>%
  group_by(contributing_factor) %>%
  summarise(
    total_collisions = n(),
    avg_casualties = mean(total_casualties, na.rm = TRUE),
    total_casualties = sum(total_casualties, na.rm = TRUE),
    .groups = 'drop'
  ) %>%
  filter(total_collisions >= 30) %>%
  arrange(desc(avg_casualties)) %>%
  head(10)

ggplot(factor_severity, aes(x = total_collisions, y = avg_casualties)) +
  geom_point(aes(size = total_casualties), color = "darkblue", alpha = 0.6) +
  scale_size_continuous(range = c(3, 15)) +
  labs(
    title = "Contributing Factor Impact Analysis",
    subtitle = "Frequency vs. Average Severity (size = total casualties)",
    x = "Number of Collisions",
    y = "Average Casualties per Collision",
    size = "Total Casualties"
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(face = "bold", size = 14)
  )

**Key Insight:** Some factors are frequent but less severe, while others are less common but more severe.


---
# Part 4: Vulnerable Road Users
## Who is most at risk?


### Question 7: Distribution of injuries among road user types


In [ ]:
# User casualties summary
user_casualties_data <- data.frame(
  User_Type = c("Pedestrians", "Cyclists", "Motorists"),
  Casualties = c(
    sum(collisions_clean$NUMBER.OF.PEDESTRIANS.INJURED + collisions_clean$NUMBER.OF.PEDESTRIANS.KILLED, na.rm = TRUE),
    sum(collisions_clean$NUMBER.OF.CYCLIST.INJURED + collisions_clean$NUMBER.OF.CYCLIST.KILLED, na.rm = TRUE),
    sum(collisions_clean$NUMBER.OF.MOTORIST.INJURED + collisions_clean$NUMBER.OF.MOTORIST.KILLED, na.rm = TRUE)
  )
)

ggplot(user_casualties_data, aes(x = reorder(User_Type, Casualties), y = Casualties, fill = User_Type)) +
  geom_bar(stat = "identity", alpha = 0.8) +
  geom_text(aes(label = Casualties), vjust = -0.5, size = 5, fontface = "bold") +
  scale_fill_manual(values = c("Pedestrians" = "#e74c3c", 
                                "Cyclists" = "#f39c12", 
                                "Motorists" = "#3498db")) +
  labs(
    title = "Total Casualties by Road User Type",
    subtitle = "Motorists account for the majority of casualties",
    x = "Road User Type",
    y = "Total Casualties (Injured + Killed)"
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    legend.position = "none"
  )

**Key Insight:** Motorists experience the highest casualty numbers, but pedestrians and cyclists remain vulnerable.


### Question 8: Vulnerable users across boroughs


In [ ]:
# Vulnerable users by borough
vulnerable_by_borough <- collisions_clean %>%
  filter(borough != "Unknown") %>%
  group_by(borough) %>%
  summarise(
    Pedestrians = sum(NUMBER.OF.PEDESTRIANS.INJURED + NUMBER.OF.PEDESTRIANS.KILLED, na.rm = TRUE),
    Cyclists = sum(NUMBER.OF.CYCLIST.INJURED + NUMBER.OF.CYCLIST.KILLED, na.rm = TRUE),
    .groups = 'drop'
  )

# Reshape for plotting
vulnerable_long <- vulnerable_by_borough %>%
  pivot_longer(cols = c(Pedestrians, Cyclists), 
               names_to = "User_Type", 
               values_to = "Casualties")

ggplot(vulnerable_long, aes(x = borough, y = Casualties, fill = User_Type)) +
  geom_bar(stat = "identity", position = "dodge", alpha = 0.8) +
  scale_fill_manual(values = c("Pedestrians" = "#e74c3c", "Cyclists" = "#f39c12")) +
  labs(
    title = "Vulnerable Road User Casualties by Borough",
    subtitle = "Pedestrian and cyclist injuries vary by location",
    x = "Borough",
    y = "Total Casualties",
    fill = "User Type"
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    legend.position = "bottom",
    axis.text.x = element_text(angle = 45, hjust = 1)
  )

**Key Insight:** Brooklyn and Manhattan show higher pedestrian casualties due to dense urban foot traffic.


---
# Part 5: Vehicle Types


### Question 9: Most common vehicle types in collisions


In [ ]:
# Top vehicle types
top_vehicles <- collisions_clean %>%
  filter(vehicle_type != "Unknown") %>%
  group_by(vehicle_type) %>%
  summarise(count = n(), .groups = 'drop') %>%
  arrange(desc(count)) %>%
  head(10)

ggplot(top_vehicles, aes(x = reorder(vehicle_type, count), y = count)) +
  geom_bar(stat = "identity", fill = "purple", alpha = 0.7) +
  geom_text(aes(label = count), hjust = -0.2, size = 3.5) +
  coord_flip() +
  labs(
    title = "Top 10 Vehicle Types in Collisions",
    subtitle = "Sedans and SUVs dominate collision statistics",
    x = "Vehicle Type",
    y = "Number of Collisions"
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(face = "bold", size = 14)
  )

**Key Insight:** Passenger vehicles dominate, reflecting NYC's vehicle composition.


---
# Part 6: Temporal Trends


### Question 10: Yearly collision trends


In [ ]:
# Yearly trends
yearly_collisions <- collisions_clean %>%
  filter(!is.na(crash_year)) %>%
  group_by(crash_year) %>%
  summarise(
    total_collisions = n(),
    .groups = 'drop'
  )

ggplot(yearly_collisions, aes(x = crash_year, y = total_collisions)) +
  geom_line(color = "darkblue", size = 1.2) +
  geom_point(color = "darkblue", size = 3) +
  geom_smooth(method = "loess", se = TRUE, color = "red", linetype = "dashed", fill = "pink", alpha = 0.3) +
  labs(
    title = "Collision Trends Over Time",
    subtitle = "Year-over-year patterns in collision frequency",
    x = "Year",
    y = "Number of Collisions"
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(face = "bold", size = 14)
  )

**Key Insight:** Time series reveals year-over-year variations requiring further investigation.


### Question 11: Monthly patterns


In [ ]:
# Monthly patterns
monthly_summary <- collisions_clean %>%
  filter(!is.na(crash_month)) %>%
  group_by(crash_month) %>%
  summarise(count = n(), .groups = 'drop')

ggplot(monthly_summary, aes(x = crash_month, y = count, group = 1)) +
  geom_line(color = "darkgreen", size = 1.2) +
  geom_point(color = "darkgreen", size = 3) +
  labs(
    title = "Seasonal Collision Patterns",
    subtitle = "Monthly distribution reveals seasonal effects",
    x = "Month",
    y = "Number of Collisions"
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    axis.text.x = element_text(angle = 45, hjust = 1)
  )

**Key Insight:** Monthly patterns may reveal seasonal effects from weather and daylight variations.


---
# Conclusions and Recommendations

## Key Findings

1. **Temporal Patterns**: Collisions peak during afternoon rush hours (3-6 PM)
2. **Geographic**: Brooklyn and Queens have highest collision frequencies
3. **Contributing Factors**: Driver inattention/distraction is the leading cause
4. **Vulnerable Users**: Pedestrians and cyclists need targeted protection
5. **Vehicle Types**: Passenger vehicles dominate collision statistics

## Recommendations

1. Increase enforcement during peak hours (3-6 PM)
2. Implement distracted driving prevention campaigns
3. Invest in protected bike lanes and pedestrian infrastructure
4. Focus interventions in Brooklyn and Queens
5. Use data-driven hotspot analysis for targeted improvements


---
## Summary Statistics


In [ ]:
# Summary statistics
cat("\n=== NYPD Motor Vehicle Collisions: Summary ===\n\n")
cat("Total Collisions:", nrow(collisions_clean), "\n")
cat("Date Range:", as.character(min(collisions_clean$crash_date, na.rm = TRUE)), 
    "to", as.character(max(collisions_clean$crash_date, na.rm = TRUE)), "\n\n")

cat("Casualties:\n")
cat("  Total Injured:", sum(collisions_clean$total_injured, na.rm = TRUE), "\n")
cat("  Total Killed:", sum(collisions_clean$total_killed, na.rm = TRUE), "\n\n")

cat("Borough Distribution:\n")
print(table(collisions_clean$borough))
